# UC10 — Asistente Inteligente de Consultas de Póliza

Asistente Q&A basado en RAG sobre documentación de pólizas usando Cortex Search.
**Valor de negocio**: Respuestas precisas en <3s para agentes del contact center.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS ASISTENTE_POLIZAS_DB;
USE SCHEMA ASISTENTE_POLIZAS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Base de Conocimiento de Pólizas

In [ ]:
CREATE OR REPLACE TABLE documentos_poliza AS
SELECT column1 AS doc_id, column2 AS producto, column3 AS seccion, column4 AS titulo, column5 AS contenido FROM VALUES
('D001','Auto','Coberturas','Responsabilidad Civil','La póliza de auto cubre la responsabilidad civil obligatoria hasta 70 millones de euros por daños corporales y 15 millones por daños materiales.'),
('D002','Auto','Coberturas','Robo','La cobertura de robo incluye el hurto total del vehículo, robo de accesorios fijos y daños durante el intento de robo. Franquicia: 150€.'),
('D003','Auto','Coberturas','Lunas','La cobertura de lunas incluye parabrisas, luneta trasera y ventanillas. Reparación sin franquicia. Sustitución con franquicia de 100€.'),
('D004','Auto','Plazos','Notificación Siniestro','El asegurado debe notificar el siniestro en un plazo máximo de 7 días hábiles. En caso de robo, denuncia policial en 24 horas.'),
('D005','Auto','Asistencia','Asistencia en Carretera','Asistencia 24h en toda España y Europa. Incluye grúa hasta 50km, vehículo de sustitución 5 días, alojamiento de emergencia.'),
('D006','Hogar','Coberturas','Incendio','Cubre daños por incendio, explosión, rayo y caída de aeronaves. Incluye continente y contenido. Valor a nuevo. Sin franquicia.'),
('D007','Hogar','Coberturas','Agua','Daños por agua: rotura de tuberías, desbordamiento, filtraciones. Incluye localización de avería y daños estéticos. Franquicia: 100€.'),
('D008','Hogar','Coberturas','Robo en Vivienda','Cobertura de robo y expoliación en domicilio. Límite por objeto: 3.000€. Requiere medidas de seguridad mínimas.'),
('D009','Hogar','Plazos','Comunicación Daños','Los daños deben comunicarse en 5 días hábiles. En caso de robo, denuncia policial inmediata y notificar en 24 horas.'),
('D010','Hogar','Coberturas','RC Hogar','RC del hogar cubre daños a terceros por el inmueble o sus ocupantes. Límite: 300.000€. Incluye daños por agua a vecinos.'),
('D011','Vida','Coberturas','Fallecimiento','Capital pagadero a beneficiarios por fallecimiento. Carencia: 12 meses para suicidio. Sin carencia para accidente.'),
('D012','Vida','Coberturas','Invalidez','Invalidez permanente absoluta: 100% del capital. Invalidez parcial: según baremo. Incluye anticipo por enfermedad grave.'),
('D013','Salud','Coberturas','Hospitalización','Hospitalización médica y quirúrgica sin límite de estancia. Habitación individual. Incluye UCI y honorarios médicos.'),
('D014','Salud','Coberturas','Consultas','Acceso a cuadro médico de especialistas sin lista de espera. Copago de 10€ por consulta.'),
('D015','Salud','Plazos','Carencias','Carencias: 6 meses hospitalización, 8 meses parto, 12 meses prótesis dentales. Sin carencia para urgencias.'),
('D016','Auto','Exclusiones','Exclusiones Auto','No se cubren: conducción bajo alcohol/drogas, uso en competición, daños intencionados, sin ITV vigente.'),
('D017','Hogar','Exclusiones','Exclusiones Hogar','No se cubren: falta de mantenimiento, humedades por condensación, plagas, desgaste natural.'),
('D018','Auto','FAQ','Cambio de Vehículo','Para cambiar vehículo, contacte con su agente o llame al 900 123 456. Sin coste si prima equivalente.'),
('D019','Hogar','FAQ','Daños por Mascota','Daños de mascotas a terceros: cubiertos por RC. Daños en la propia vivienda: NO cubiertos.'),
('D020','Salud','FAQ','Segunda Opinión','Servicio de segunda opinión médica para diagnósticos graves. Sin coste adicional.')
;

SELECT * FROM documentos_poliza LIMIT 5;

---

## Paso 3: Crear Chunks para Indexación

In [ ]:
CREATE OR REPLACE TABLE chunks_poliza AS
SELECT doc_id || '_C1' AS chunk_id, doc_id, producto, seccion, contenido AS chunk_texto, 1 AS chunk_orden
FROM documentos_poliza;

SELECT COUNT(*) AS total_chunks FROM chunks_poliza;

---

## Paso 4: Crear Cortex Search Service

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE buscador_polizas
    ON chunk_texto
    ATTRIBUTES producto, seccion
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (SELECT chunk_texto, producto, seccion, chunk_id, doc_id FROM chunks_poliza);

---

## Paso 5: Implementar Pipeline RAG

In [ ]:
CREATE OR REPLACE FUNCTION responder_consulta(pregunta VARCHAR)
RETURNS VARCHAR LANGUAGE SQL AS $$
    SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Eres un asistente experto en seguros. Responde basándote SOLO en el contexto. Si no encuentras la respuesta, di "No tengo información suficiente".\n\nContexto:\n' ||
        (SELECT LISTAGG(chunk_texto, '\n---\n') FROM chunks_poliza WHERE LOWER(chunk_texto) LIKE '%' || LOWER(SPLIT_PART(pregunta,' ',1)) || '%' LIMIT 3) ||
        '\n\nPregunta: ' || pregunta || '\n\nRespuesta:')
$$;

SELECT responder_consulta('¿Qué cubre el seguro de hogar en caso de robo?') AS respuesta;

---

## Paso 6: Probar con Preguntas Tipo

In [ ]:
SELECT 'Lunas' AS tema, responder_consulta('¿La rotura de cristales tiene franquicia?') AS respuesta
UNION ALL SELECT 'Plazo', responder_consulta('¿Cuál es el plazo para notificar un siniestro de auto?')
UNION ALL SELECT 'Mascota', responder_consulta('¿Están cubiertos los daños por mascota?')
UNION ALL SELECT 'Asistencia', responder_consulta('¿Cómo solicito asistencia en carretera?')
UNION ALL SELECT 'Carencias', responder_consulta('¿Cuánto tiempo de carencia tiene la hospitalización?');

---

## Paso 7: Dashboard Interactivo

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Asistente de Consultas de Póliza')
st.markdown('### Respuestas IA para Agentes — Cortex Search + RAG')

producto = st.selectbox('Producto', ['Todos','Auto','Hogar','Vida','Salud'])
pregunta = st.text_input('Pregunta del agente:', placeholder='Ej: ¿Qué cubre la póliza de hogar en caso de incendio?')

if pregunta:
    with st.spinner('Buscando respuesta...'):
        result = session.sql(f"SELECT responder_consulta('{pregunta}') AS resp").collect()
        st.markdown('### Respuesta')
        st.write(result[0]['RESP'])

st.markdown('---')
st.subheader('Base de Conocimiento')
docs = session.table('documentos_poliza').to_pandas()
if producto != 'Todos': docs = docs[docs['PRODUCTO']==producto]
st.dataframe(docs[['DOC_ID','PRODUCTO','SECCION','TITULO']], use_container_width=True, height=300)
st.caption('Desarrollado con Snowflake Cortex Search + Streamlit')

---

## Paso 8: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS ASISTENTE_POLIZAS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;